[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Connections and Cursors


## What you will be able to do

Open a connection to a SQLite database and close it when you are done, run statements through a
cursor, and read the results a row at a time, a few rows at a time, or all at once, knowing what
each way holds in memory. Open a database that a program may only read, give every thread a
connection of its own, and find the connection a program forgot to close.


## The idea

### The problem

The **Why sqlite3** notebook asked the stations' year a handful of questions, and closed every
connection it opened. Programs that outgrow a notebook are rarely that tidy. A report function opens
the database, runs its query and returns the answer, and nothing closes the connection, so a
scheduler that runs it every minute leaves a connection behind every minute. A web page asks for a
station's readings with `fetchall`, and every request builds a list of 8,760 rows in order to show
the first twenty. A summary loops over the stations, reuses its cursor for a count, and prints Bergen
alone, with no error. A second thread, started to keep the page responsive, borrows the main
thread's connection and fails.

Each of these treats a connection or a cursor as an ordinary value, like a list or a string. Neither
is one. A connection holds an open database until something closes it, and a cursor holds its place
in one statement's results. Neither shows in a program's output until something goes wrong.

### What a connection and a cursor are

> A **connection** is an open database: the file, SQLite's working state for it, and any
> transaction in progress, held by one `sqlite3.Connection` object until its `close` method releases
> them. A **cursor** runs a statement on a connection and hands back the result:
> `cursor.execute(sql)` runs it, and `fetchone`, `fetchmany` and `fetchall` take the next row, the
> next few rows, or every row that is left. A cursor reads its result from SQLite as you fetch it, so
> a row it has handed back is gone from it, and executing another statement on the same cursor
> throws away the rows it had not handed back yet. `connection.execute` is a shortcut that makes a
> new cursor, runs the statement on it and returns that cursor, which is how the **Why sqlite3**
> notebook ran its queries without making a cursor itself.

### Why it works that way

- **A connection is a resource, not a value.** It keeps the database file open and memory allocated
  inside SQLite until `close` runs. Python closes a connection that nothing refers to any more, and
  from Python 3.13 it emits a warning when it has to, which most programs never display.
- **`with` on a connection is about transactions, not about closing.** The block commits when it
  finishes and rolls back when it raises, which the **Transactions** notebook takes apart, and the
  connection stays open afterward. `closing`, from `contextlib`, is what closes it.
- **A cursor hands rows over as you ask for them.** Looping over a cursor, or calling `fetchone`,
  holds one row at a time, while `fetchall` builds a list of every row the statement returns. That
  list is right for a few rows and wrong for a whole table.
- **A cursor holds one statement's results at a time.** Executing a new statement on it replaces
  the rows it had not handed back, so a statement whose results are still being read needs a cursor
  of its own.
- **A cursor you stop reading still holds the database.** Its statement stays open, with a read
  lock on the file, until the cursor is read to the end, closed or deleted, and a write from another
  connection waits for it. Closing the connection does not release the lock while the cursor lives.
- **`rowcount` counts changes, not results.** After `INSERT`, `UPDATE` or `DELETE` it holds how many
  rows changed. After `SELECT` it holds -1, because a query changes no rows.
- **A connection belongs to the thread that made it.** By default sqlite3 refuses to let a
  connection made in one thread run statements in another, so a program that uses threads gives
  every thread a connection of its own.
- **Every connection to `:memory:` is a separate database.** It lives exactly as long as that
  connection, which makes it ideal for a quick experiment and useless for sharing.

### Where you will meet this

Connections and cursors are not SQLite's invention. Python's database API, set out in PEP 249,
gives every database driver a `connect` function, and connection and cursor objects with `execute`,
`fetchone`, `fetchmany` and `fetchall`, so psycopg, the PostgreSQL driver in the **asyncpg and
psycopg3, Deep Dive** guide, reads the way this notebook does. The engine in the **SQLAlchemy, Deep
Dive** guide keeps a pool of these connections and lends them out. A web application opens a
connection when a request arrives and closes it when the response goes out, and a program that works
in threads keeps one connection per thread. The **Files, Paths and Formats** guide closed files with
`with open(...)`, and a connection is the case where `with` does something else.

### What this notebook covers

- A connection, opened, used and closed
- A cursor, and the statement it runs
- `fetchone`, `fetchmany` and `fetchall`, and looping over a cursor
- What each way of fetching holds in memory
- What a cursor reports about its results
- `with` on a connection, and `closing` from `contextlib`
- A connection that can only read
- A database held in memory
- When to make a cursor yourself, and when to let `conn.execute` make one
- A report that reads every station a row at a time, and closes what it opens
- Seven errors: a cursor whose connection has closed, a connection nobody closed, a second connection
  to `:memory:`, a write blocked by a cursor left half read, a connection used from another thread, a
  cursor reused inside its own loop, and `rowcount` read after a `SELECT`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3
from contextlib import closing

with closing(sqlite3.connect(":memory:")) as conn:
    conn.execute("CREATE TABLE readings (station TEXT, celsius REAL)")
    conn.executemany("INSERT INTO readings VALUES (?, ?)",
                     [("Bergen", 4.2), ("Oslo", -2.4), ("Svalbard", -18.5), ("Tromso", -7.9)])
    cursor = conn.execute("SELECT station, celsius FROM readings ORDER BY celsius")
    print("columns:", [column[0] for column in cursor.description])
    print("fetchone:", cursor.fetchone())
    print("fetchmany(2):", cursor.fetchmany(2))
    print("fetchall:", cursor.fetchall())
    print("fetchone after the end:", cursor.fetchone())

try:
    conn.execute("SELECT 1")
except sqlite3.ProgrammingError as error:
    print("after the with block:", error)
```

```
columns: ['station', 'celsius']
fetchone: ('Svalbard', -18.5)
fetchmany(2): [('Tromso', -7.9), ('Oslo', -2.4)]
fetchall: [('Bergen', 4.2)]
fetchone after the end: None
after the with block: Cannot operate on a closed database.
```

One connection, one cursor, and every way of reading its rows. Each fetch took up where the last one
stopped, and once the rows ran out, `fetchone` answered `None`. `closing` closed the connection when
the block ended, so the statement after the block had nothing to run on.


## Setup

Eleven imports, and the year of readings, written straight into a database this time.

- `sqlite3` opens connections and runs statements
- `closing`, from `contextlib`, closes a connection at the end of a `with` block
- `math`, `datetime` and `timedelta` make the same year of readings the **Why sqlite3** notebook made
- `Path` names the scratch folder and the database in it
- `tracemalloc` measures the most memory a way of fetching holds
- `gc` and `warnings` catch the warning a forgotten connection emits, in Common errors
- `re` replaces a memory address and thread numbers in two messages, since both change on every run
- `threading` runs statements from other threads, in Common errors
- `shutil` removes the scratch folder at the end

Setup builds `scratch/stations.db` with the table of readings that the **Why sqlite3** notebook
loaded from its CSV, hour for hour and value for value, by handing a generator straight to
`executemany`.


In [1]:
import gc
import math
import re
import shutil
import sqlite3
import threading
import tracemalloc
import warnings
from contextlib import closing
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


DATABASE.unlink(missing_ok=True)
build = sqlite3.connect(DATABASE)
build.execute("CREATE TABLE readings (station TEXT NOT NULL, hour TEXT NOT NULL, celsius REAL)")
build.executemany("INSERT INTO readings (station, hour, celsius) VALUES (?, ?, ?)", year_of_readings())
build.commit()
build.close()

print("built", DATABASE)


built scratch/stations.db


## Worked examples

### A connection, opened, used and closed

`sqlite3.connect` returns a connection, and every statement runs through one. The connection stays
open, holding the database file, until its `close` method runs. Closing is safe to repeat, which lets
cleanup code call `close` without first checking whether something else already has:


In [2]:
conn = sqlite3.connect(DATABASE)
print("a", type(conn).__name__, "to", DATABASE.name)
print("readings:", conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0])

conn.close()
conn.close()
print("closed twice, and nothing raised")


a Connection to stations.db
readings: 35040
closed twice, and nothing raised


A closed connection refuses every statement, which is where the Common errors section of this
notebook begins.

### A cursor, and the statement it runs

A connection makes cursors with its `cursor` method. `execute` runs a statement on the cursor and
returns the cursor itself, so a fetch can follow on the same line. The cursor remembers its
connection, and its `description` names the columns of the result, with one tuple per column: the
name first, then six places that sqlite3 leaves as `None`:


In [3]:
conn = sqlite3.connect(DATABASE)
cursor = conn.cursor()

result = cursor.execute("SELECT station, hour, celsius FROM readings WHERE station = ? ORDER BY hour", ("Oslo",))
print("execute returned the cursor:", result is cursor)
print("its connection:", cursor.connection is conn)
print("description:", cursor.description)
print("first row:", cursor.fetchone())


execute returned the cursor: True
its connection: True
description: (('station', None, None, None, None, None, None), ('hour', None, None, None, None, None, None), ('celsius', None, None, None, None, None, None))
first row: ('Oslo', '2025-01-01T00:00', -3.5)


`conn.execute` does that work in one call: it makes a new cursor, executes the statement on it and
returns it. Every call makes another cursor, and a cursor keeps its own place in its own results, so
the count below leaves the Oslo cursor where it was:


In [4]:
count = conn.execute("SELECT COUNT(*) FROM readings WHERE station = ?", ("Oslo",))
print("a cursor of its own:", count is not cursor)
print("Oslo's hours:", count.fetchone()[0])
print("the first cursor carries on:", cursor.fetchone())
cursor.close()


a cursor of its own: True
Oslo's hours: 8760
the first cursor carries on: ('Oslo', '2025-01-01T01:00', -5.3)


`close` ends a cursor's statement. The Oslo cursor stopped after two of its 8,760 rows, and a cursor
stopped partway keeps a read lock on the database until it is closed, read to the end or deleted,
which one of the Common errors below shows going wrong.

### fetchone, fetchmany and fetchall

Three ways to take rows from a cursor. `fetchone` takes the next row, `fetchmany(n)` the next `n`
rows as a list, and `fetchall` every row that is left, and each starts where the last one stopped.
Midsummer's day at Bergen is 24 rows in hour order:


In [5]:
day = conn.execute("SELECT hour, celsius FROM readings WHERE station = ? AND hour LIKE ? ORDER BY hour",
                   ("Bergen", "2025-06-21%"))

print("fetchone:     ", day.fetchone())
print("fetchmany(3): ", day.fetchmany(3))
print("fetchall:     ", len(day.fetchall()), "rows")
print("fetchone now: ", day.fetchone())
print("fetchall now: ", day.fetchall())


fetchone:      ('2025-06-21T00:00', 13.4)
fetchmany(3):  [('2025-06-21T01:00', 13.3), ('2025-06-21T02:00', 13.3), ('2025-06-21T03:00', 13.5)]
fetchall:      20 rows
fetchone now:  None
fetchall now:  []


Once the rows run out, `fetchone` returns `None` and `fetchall` an empty list, and neither raises.
A cursor is also an iterator: a `for` loop over it takes one row at a time until the rows run out.
Like any iterator it runs out once, so a second loop over the same cursor finds nothing:


In [6]:
warmest = conn.execute("""
    SELECT hour, celsius FROM readings
    WHERE station = ? AND hour LIKE ?
    ORDER BY celsius DESC, hour
    LIMIT 3
""", ("Bergen", "2025-06-21%"))

for hour, celsius in warmest:
    print(hour, celsius)
print("a second loop over the same cursor:", [hour for hour, celsius in warmest])


2025-06-21T15:00 19.7
2025-06-21T14:00 19.3
2025-06-21T13:00 18.7
a second loop over the same cursor: []


### What each way of fetching holds

`fetchall` and a loop differ in where the rows go. `fetchall` builds a list holding a tuple for every
row, while a loop holds one row, which the next row replaces. `tracemalloc` records the most memory
Python held while each one ran, here finding the coldest reading of the year both ways:


In [7]:
tracemalloc.start()
rows = conn.execute("SELECT celsius FROM readings WHERE celsius IS NOT NULL").fetchall()
coldest_from_list = min(celsius for (celsius,) in rows)
peak_list = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()
del rows

tracemalloc.start()
coldest_from_loop = None
for (celsius,) in conn.execute("SELECT celsius FROM readings WHERE celsius IS NOT NULL"):
    if coldest_from_loop is None or celsius < coldest_from_loop:
        coldest_from_loop = celsius
peak_loop = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

print("the same answer both ways:", coldest_from_list, coldest_from_loop)
print("fetchall held more than 1 MB at its peak:", peak_list > 1_000_000)
print("the loop held less than 100 KB at its peak:", peak_loop < 100_000)


the same answer both ways: -17.3 -17.3
fetchall held more than 1 MB at its peak: True
the loop held less than 100 KB at its peak: True


The list of every reading took megabytes, and the loop stayed under a tenth of one. The exact
figures depend on the Python running them, so the cell prints only which side of those lines they
fell. For this question `SELECT MIN(celsius)` would be better than either, since SQLite would send
back one number instead of every reading. A loop over a cursor is for work that SQL cannot easily
express, like the report at the end of these examples.

### What a cursor reports

`rowcount` holds how many rows the last `INSERT`, `UPDATE` or `DELETE` changed. An `UPDATE` that sets
every Bergen reading on midsummer's day to the value it already has changes no temperature and counts
24 rows. `rollback` throws the change away, which the **Transactions** notebook explains. After a
`SELECT`, `rowcount` is -1, which one of the Common errors below turns into a bug:


In [8]:
update = conn.execute("UPDATE readings SET celsius = celsius WHERE station = ? AND hour LIKE ?",
                      ("Bergen", "2025-06-21%"))
print("rowcount after the UPDATE:", update.rowcount)
conn.rollback()
conn.close()


rowcount after the UPDATE: 24


### with, and closing

A connection works as a context manager, and what that does is easy to guess wrong. `with conn:`
commits when the block finishes and rolls back when it raises, which the **Transactions** notebook
explains, and the connection is still open when the block ends:


In [9]:
with sqlite3.connect(DATABASE) as conn:
    before = conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0]

print("still answering after the block:", conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0] == before)
conn.close()


still answering after the block: True


`closing`, from `contextlib`, calls `close` when its block ends, however it ends. Written together,
with `closing` first and the connection second, one block commits its work and then closes the
connection:


In [10]:
with closing(sqlite3.connect(DATABASE)) as conn, conn:
    conn.execute("CREATE TABLE stations (name TEXT NOT NULL, latitude REAL NOT NULL)")
    conn.executemany("INSERT INTO stations (name, latitude) VALUES (?, ?)",
                     [("Bergen", 60.39), ("Oslo", 59.91), ("Svalbard", 78.22), ("Tromso", 69.65)])

with closing(sqlite3.connect(DATABASE)) as check:
    print("committed, then closed:", check.execute("SELECT name FROM stations ORDER BY name").fetchall())


committed, then closed: [('Bergen',), ('Oslo',), ('Svalbard',), ('Tromso',)]


The second connection found every station, so the first block committed before it closed.

### A connection that can only read

A report has no business changing the data it reports on. Opened by a URI with `mode=ro`, a
connection reads as usual and refuses every change, which turns a mistake in a report into an error
rather than damage:


In [11]:
reader = sqlite3.connect(f"file:{DATABASE}?mode=ro", uri=True)
print("reads:", reader.execute("SELECT COUNT(*) FROM readings").fetchone()[0], "readings")

try:
    reader.execute("DELETE FROM readings WHERE station = ?", ("Oslo",))
except sqlite3.OperationalError as error:
    print("refuses to change them:", error)
reader.close()


reads: 35040 readings
refuses to change them: attempt to write a readonly database


The **Why sqlite3** notebook opened a database with `mode=rw`, which never creates a file. `mode=ro`
goes a step further, and never writes to one.

### A database held in memory

The name `:memory:` opens a database that lives in memory instead of a file. It is the quickest place
to try out a statement, and it disappears when its connection closes:


In [12]:
with closing(sqlite3.connect(":memory:")) as memory:
    memory.execute("CREATE TABLE notes (text TEXT)")
    memory.execute("INSERT INTO notes (text) VALUES (?)", ("a table that never touches the disk",))
    print(memory.execute("SELECT text FROM notes").fetchall())

print("files in scratch:", sorted(path.name for path in SCRATCH.iterdir()))


[('a table that never touches the disk',)]
files in scratch: ['stations.db']


### A cursor of your own, or conn.execute

This notebook has run statements two ways: on a cursor made with `conn.cursor()`, and through
`conn.execute`, which makes the cursor itself. They are the same machinery. `conn.execute` creates a
new cursor, runs the statement on it and returns it, so what comes back is an ordinary cursor, with
every method and attribute of a cursor made by hand:


In [13]:
conn = sqlite3.connect(DATABASE)

made_by_hand = conn.cursor()
made_by_hand.execute("SELECT hour, celsius FROM readings WHERE station = ? ORDER BY hour", ("Tromso",))
made_by_execute = conn.execute("SELECT hour, celsius FROM readings WHERE station = ? ORDER BY hour", ("Tromso",))

print("both are cursors:", type(made_by_hand).__name__, type(made_by_execute).__name__)
print("the same first row:", made_by_hand.fetchone() == made_by_execute.fetchone())
print("the same columns:", made_by_hand.description == made_by_execute.description)
made_by_execute.arraysize = 3                 # a setting works on the cursor that execute returned, too
print("fetchmany on that cursor:", made_by_execute.fetchmany())

made_by_hand.close()
made_by_execute.close()
conn.close()


both are cursors: Cursor Cursor
the same first row: True
the same columns: True
fetchmany on that cursor: [('2025-01-01T01:00', -8.5), ('2025-01-01T02:00', -8.5), ('2025-01-01T03:00', -8.3)]


Both cursors are the same kind of object, reading the same rows the same way, and the cursor that
`conn.execute` returned took a setting and a `fetchmany` like any other. So the choice between them is
not about what a cursor can do. It is about how many statements share one cursor, and about whether
the code has to work beyond sqlite3:

| Write | When | Why |
|---|---|---|
| `conn.execute(...)` | almost always: one statement, with its rows read at once or through the cursor it returns | every call gets a fresh cursor, so no statement can throw away rows that another has not finished reading |
| `cursor = conn.cursor()`, then `cursor.execute(...)` | code that has to run on any Python database driver, not only sqlite3 | PEP 249 gives a connection a `cursor` method and no `execute`, and not every driver adds the shortcut: psycopg 3 has one, and psycopg2 does not |
| one cursor, with `execute` called on it again | statements that run one after another, each read to the end before the next, such as a function handed a cursor to write with | a cursor holds one statement's results at a time, so it can be reused only once nothing is still reading it |

The Python documentation takes the side of the first row: its shortcut methods exist so that you need
not create cursor objects it calls "often superfluous". A cursor made by hand earns its place in code
that must also run on a driver without the shortcut. Where it goes wrong is being executed again while
its rows are still being read, which one of the Common errors below shows stopping a report after one
station. And whichever way a cursor was made, one you stop reading partway is one to close.


### A report, a row at a time

The pieces of this notebook in one job: the longest spell of consecutive hours below freezing at
every station, which SQL cannot easily express and a loop over a cursor can. One read-only
connection, closed by `closing`, serves the whole report. The stations are read into a list before
the loop, and every statement gets a cursor of its own, so no loop reads from a cursor that another
statement has taken over:


In [14]:
def longest_freeze(conn, station):
    """The most consecutive hours below freezing at a station, reading one row at a time.
    A missing reading ends a spell, since nobody knows whether that hour froze."""
    longest = current = 0
    for (celsius,) in conn.execute("SELECT celsius FROM readings WHERE station = ? ORDER BY hour", (station,)):
        current = current + 1 if celsius is not None and celsius < 0 else 0
        longest = max(longest, current)
    return longest


with closing(sqlite3.connect(f"file:{DATABASE}?mode=ro", uri=True)) as conn:
    stations = [name for (name,) in conn.execute("SELECT DISTINCT station FROM readings ORDER BY station")]
    for station in stations:
        print(f"{station:<9} longest freeze: {longest_freeze(conn, station):>5} hours")


Bergen    longest freeze:    15 hours
Oslo      longest freeze:    21 hours
Svalbard  longest freeze:  1953 hours
Tromso    longest freeze:  1287 hours


The loop read every reading of the year, one row at a time, and held no list of them. The
connection closed as the block ended, and it could not have changed a row while it was open.

### Where each part came from

| In the report | What it relies on | The section that showed it |
|---|---|---|
| `closing(sqlite3.connect(...))` | a connection closed when the block ends, however it ends | with, and closing |
| `file:...?mode=ro` with `uri=True` | a connection that reads and refuses every change | A connection that can only read |
| `conn.execute` inside `longest_freeze` | a fresh cursor for every statement, so no loop's rows are thrown away | A cursor of your own, or conn.execute |
| `for (celsius,) in conn.execute(...)` | a cursor that hands over one row at a time | fetchone, fetchmany and fetchall |
| no list of readings anywhere | a loop that holds one row, where `fetchall` holds them all | What each way of fetching holds |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/02-connections-and-cursors-solutions.ipynb).

**1.** Open a connection with `closing`, print how many different stations the readings come from,
and let the block close the connection.


In [15]:
# your code here


**2.** On one cursor over Tromso's readings in hour order, print the first three rows one at a time
with `fetchone`, then the next two together with `fetchmany`.


In [16]:
# your code here


**3.** On a cursor over Svalbard's readings, print what `fetchmany()` returns with no number. Then
set the cursor's `arraysize` to 4, and print what `fetchmany()` returns now.


In [17]:
# your code here


**4.** Print the column names of
`SELECT station, MIN(celsius) AS coldest, MAX(celsius) AS warmest FROM readings GROUP BY station`,
taken from the cursor before fetching a single row.


In [18]:
# your code here


**5.** Write a function `first_hour_below(database, station, celsius)` that opens and closes its own
connection and returns the first hour a station's reading fell below a temperature, or `None` if it
never did. Print it for Oslo below -5, and for Bergen below -20.


In [19]:
# your code here


**6.** Through one read-only connection, reading the rows one at a time, find the biggest rise in
temperature from one hour to the next at any station, and print the rise, the station and the hour.


In [20]:
# your code here


## Common errors

### sqlite3.ProgrammingError: Cannot operate on a closed database.


In [21]:
def readings_for(station):
    """A station's readings, as a cursor to loop over."""
    with closing(sqlite3.connect(DATABASE)) as conn:
        return conn.execute("SELECT hour, celsius FROM readings WHERE station = ? ORDER BY hour", (station,))


for hour, celsius in readings_for("Oslo"):
    print(hour, celsius)


ProgrammingError: Cannot operate on a closed database.

`readings_for` returned a cursor, and `closing` closed that cursor's connection as the function
returned, before the loop had asked for a single row. A cursor reads from its connection as you
fetch, so it cannot outlive the connection. Either read the rows inside the block and return them,
or let the caller keep the connection open and pass it in:


In [22]:
def first_readings(conn, station, count):
    """A station's first readings, from a connection the caller keeps open."""
    return conn.execute("SELECT hour, celsius FROM readings WHERE station = ? ORDER BY hour", (station,)).fetchmany(count)


with closing(sqlite3.connect(DATABASE)) as conn:
    for hour, celsius in first_readings(conn, "Oslo", 3):
        print(hour, celsius)


2025-01-01T00:00 -3.5
2025-01-01T01:00 -5.3
2025-01-01T02:00 -5.3


### ResourceWarning: unclosed database


In [23]:
def count_readings(database):
    """How many readings the database holds."""
    conn = sqlite3.connect(database)
    return conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0]


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", ResourceWarning)      # Python hides this warning unless asked
    print("readings:", count_readings(DATABASE))
    gc.collect()

for warning in caught:
    print(f"{warning.category.__name__}: {re.sub(r'0x[0-9a-fA-F]+', '0x...', str(warning.message))}")


readings: 35040


`count_readings` returned its answer and dropped the only reference to its connection, so Python
closed the connection itself, and from Python 3.13 on it says so with this warning. Python's default
settings, and a notebook's, hide the warning, so the cell asks for it with `simplefilter`. On an
earlier Python the cell prints only the count, and the connection closes just as silently. The
address in the message is where the connection sat in memory, which changes on every run, so the
cell prints `0x...` in its place. Nothing failed here, but a program built this way leaves every
connection open until Python happens to collect it. Close what you open:


In [24]:
def count_readings(database):
    """How many readings the database holds, with the connection closed before the function returns."""
    with closing(sqlite3.connect(database)) as conn:
        return conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0]


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", ResourceWarning)
    print("readings:", count_readings(DATABASE))
    gc.collect()

print("warnings:", len(caught))


readings: 35040
warnings: 0


### sqlite3.OperationalError: no such table: notes


In [25]:
first = sqlite3.connect(":memory:")
first.execute("CREATE TABLE notes (text TEXT)")

second = sqlite3.connect(":memory:")
second.execute("SELECT text FROM notes")


OperationalError: no such table: notes

`:memory:` is not the name of one shared database. Every connection to it gets a new, empty database
of its own, so `second` has never seen the table that `first` made. To share a table, share the
connection that holds it, or put the database in a file that both connections open:


In [26]:
second.close()

first.execute("INSERT INTO notes (text) VALUES (?)", ("written and read on one connection",))
print(first.execute("SELECT text FROM notes").fetchall())
first.close()


[('written and read on one connection',)]


### sqlite3.OperationalError: database is locked


In [27]:
preview = sqlite3.connect(DATABASE)
first_hours = preview.execute("SELECT hour, celsius FROM readings WHERE station = ? ORDER BY hour", ("Oslo",))
print("a preview:", first_hours.fetchmany(3))

writer = sqlite3.connect(DATABASE, timeout=1)       # wait a second for the lock, rather than the default five
writer.execute("UPDATE stations SET latitude = ? WHERE name = ?", (69.65, "Tromso"))
writer.commit()


a preview: [('2025-01-01T00:00', -3.5), ('2025-01-01T01:00', -5.3), ('2025-01-01T02:00', -5.3)]


OperationalError: database is locked

`fetchmany(3)` took three of Oslo's 8,760 readings and stopped, so the preview's statement is still
open, holding a read lock on the file in case it is asked for the rest. The writer's `UPDATE` went
through, but saving it needs the file to itself, so `commit` waited the second that `timeout=1`
allowed, and gave up. Closing the preview's connection would not have helped: a connection closed
while one of its cursors is half read keeps the file locked until that cursor is closed or deleted.
The **Concurrency and WAL** notebook explains the locks themselves. Close a cursor you stop reading,
and the change the writer is holding can be committed:


In [28]:
first_hours.close()
writer.commit()

(latitude,) = writer.execute("SELECT latitude FROM stations WHERE name = ?", ("Tromso",)).fetchone()
print("committed, and Tromso's latitude is", latitude)
writer.close()
preview.close()


committed, and Tromso's latitude is 69.65


### sqlite3.ProgrammingError: SQLite objects created in a thread can only be used in that same thread


In [29]:
conn = sqlite3.connect(DATABASE)
failures = []


def count_in_the_background():
    try:
        conn.execute("SELECT COUNT(*) FROM readings")
    except sqlite3.ProgrammingError as error:
        failures.append(error)


worker = threading.Thread(target=count_in_the_background)
worker.start()
worker.join()

for error in failures:
    print(f"{type(error).__module__}.{type(error).__name__}: {re.sub(r'[0-9]+', 'N', str(error))}")


sqlite3.ProgrammingError: SQLite objects created in a thread can only be used in that same thread. The object was created in thread id N and this is thread id N.


The connection was made in the notebook's own thread, and `count_in_the_background` ran in another,
so sqlite3 refused before SQLite saw the statement. An exception raised in a thread never reaches the
cell that started the thread, so the function catches it and the cell prints it, with the thread
numbers, which change on every run, replaced by `N`. `check_same_thread=False` lifts the check, and
then the program itself has to keep two threads from using the connection at once. It is simpler to
give every thread a connection of its own:


In [30]:
conn.close()
counts = {}


def count_station(station):
    with closing(sqlite3.connect(DATABASE)) as own:
        counts[station] = own.execute("SELECT COUNT(*) FROM readings WHERE station = ?", (station,)).fetchone()[0]


workers = [threading.Thread(target=count_station, args=(station,)) for station in STATIONS]
for worker in workers:
    worker.start()
for worker in workers:
    worker.join()

print(sorted(counts.items()))


[('Bergen', 8760), ('Oslo', 8760), ('Svalbard', 8760), ('Tromso', 8760)]


### No error, and a report that stops after one station: a cursor reused inside its own loop


In [31]:
conn = sqlite3.connect(DATABASE)
cursor = conn.cursor()

for (station,) in cursor.execute("SELECT DISTINCT station FROM readings ORDER BY station"):
    cursor.execute("SELECT COUNT(*) FROM readings WHERE station = ? AND celsius < 0", (station,))
    print(f"{station:<9} {cursor.fetchone()[0]:>5} hours below freezing")


Bergen     1220 hours below freezing


Four stations, and one line. The loop reads station names from `cursor`, and on its first pass the
count ran on that same cursor, which threw away the three names the loop had not read yet. `fetchone`
took the count, the loop asked the cursor for its next station, found nothing left, and ended as if
the report were done. Give the inner statement a cursor of its own:


In [32]:
for (station,) in cursor.execute("SELECT DISTINCT station FROM readings ORDER BY station"):
    below = conn.execute("SELECT COUNT(*) FROM readings WHERE station = ? AND celsius < 0", (station,)).fetchone()[0]
    print(f"{station:<9} {below:>5} hours below freezing")
conn.close()


Bergen     1220 hours below freezing
Oslo       1848 hours below freezing
Svalbard   5871 hours below freezing
Tromso     3203 hours below freezing


### No error, and a query that found -1 rows: rowcount read after a SELECT


In [33]:
def missing_hours(conn, station):
    """How many hours a station sent no reading."""
    cursor = conn.execute("SELECT hour FROM readings WHERE station = ? AND celsius IS NULL", (station,))
    return cursor.rowcount


with closing(sqlite3.connect(DATABASE)) as conn:
    for station in STATIONS:
        print(f"{station:<9} {missing_hours(conn, station)} hours missing")


Bergen    -1 hours missing
Oslo      -1 hours missing
Svalbard  -1 hours missing
Tromso    -1 hours missing


Every station reports -1, Svalbard's silent day included. `rowcount` counts the rows a statement
changed, a `SELECT` changes none, and sqlite3 sets it to -1 whatever the query found, which also
makes `if cursor.rowcount:` true for a query with no results at all. Count with SQL, or count the
rows you fetched:


In [34]:
def missing_hours(conn, station):
    """How many hours a station sent no reading, counted by SQLite."""
    return conn.execute("SELECT COUNT(*) FROM readings WHERE station = ? AND celsius IS NULL", (station,)).fetchone()[0]


with closing(sqlite3.connect(DATABASE)) as conn:
    for station in STATIONS:
        print(f"{station:<9} {missing_hours(conn, station)} hours missing")


Bergen    0 hours missing
Oslo      0 hours missing
Svalbard  24 hours missing
Tromso    0 hours missing


Last, the notebook is finished with its files, so this cell removes the scratch folder, with the
database in it:


In [35]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A connection holds an open database until `close` runs, and closing it twice is harmless.
- `with conn:` commits or rolls back and leaves the connection open, and `closing` is what closes it
  when a block ends.
- A cursor holds one statement's results: `fetchone`, `fetchmany` and `fetchall` take up where the
  last fetch stopped, and a new statement on the same cursor throws away the rest, so a statement
  still being read needs a cursor of its own.
- `conn.execute` gives every statement a fresh cursor, which suits almost all sqlite3 code. A cursor
  made with `conn.cursor()` is for code that must run on any Python database driver, and one cursor
  serves several statements only when each is read to the end before the next.
- A cursor you stop reading keeps the database locked for writers until it is closed, read to the
  end or deleted, even after its connection is closed.
- A loop over a cursor holds one row at a time and `fetchall` holds every row, so a loop suits a
  whole table and `fetchall` a few rows.
- `description` names a result's columns, and `rowcount` counts changed rows, holding -1 after a
  `SELECT`.
- A connection opened with `mode=ro` cannot change the database, every `:memory:` connection has a
  database of its own, and every thread needs a connection of its own.


## What is next

The **Tables and Queries** notebook turns from the objects to the SQL they run: creating tables,
selecting rows from them, and joining one table to another.


---

&#8592; **Previous:** [Why sqlite3](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/01-why-sqlite3.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
